<a href="https://colab.research.google.com/github/CharuGurusamy/CharuGurusamy/blob/main/Inference_Day_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate


### Log in to Hugging Face

To access gated models like `google/gemma-2-2b-it`, you need to authenticate with Hugging Face. Follow the steps mentioned above to get your token and add it to Colab secrets as `HF_TOKEN`.

In [1]:
# Import necessary libraries
from huggingface_hub import login
from google.colab import userdata

# Retrieve the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")


Successfully logged in to Hugging Face!


After successfully logging in, you can now re-run the cell above to load the model.

In [15]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Keep the prompt extremely simple to trace the tokens cleanly
prompt = "Who are you."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda") # Changed back to CUDA
print(f"IInput_Id : '{input_ids}'")
max_new_tokens = 5
current_inputs = input_ids

print(f"Starting prompt: '{prompt}'\n")
print(f"{'Chosen Token':<15} | {'Probability':<12} | {'Top Alternatives Considered'}")
print("-" * 70)

# Manually loop through each token generation step
for _ in range(max_new_tokens):
    with torch.no_grad():
        # Get raw model outputs (logits)
        outputs = model(current_inputs)
        next_token_logits = outputs.logits[:, -1, :]
        #print(f"next tokdn logits : '{next_token_logits}'")
        # Convert logits into absolute probabilities (0.0 to 1.0)
        probs = F.softmax(next_token_logits, dim=-1).squeeze()

        # Extract the top 4 highest-probability choices
        top_probs, top_indices = torch.topk(probs, k=4)

        # The model selects the highest probability token (index 0)
        chosen_idx = top_indices[0].item()
        chosen_prob = top_probs[0].item()
        chosen_token = tokenizer.decode([chosen_idx])

        # Format the alternative choices for display
        alternatives = []
        for p, idx in zip(top_probs[1:], top_indices[1:]):
            alt_token = tokenizer.decode([idx.item()])
            alternatives.append(f"'{alt_token.strip()}' ({p.item()*100:.1f}%)")
        alt_str = ", ".join(alternatives)

        # Print the decision matrix for this step
        print(f"{f"'{chosen_token.strip()}'":<15} | {chosen_prob*100:<10.1f}% | {alt_str}")

        # Append chosen token to inputs for the next iteration step
        current_inputs = torch.cat([current_inputs, top_indices[0].unsqueeze(0).unsqueeze(0)], dim=-1)

        if chosen_idx == tokenizer.eos_token_id:
            print("\n[End of Text Token Reached]")
            break

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

IInput_Id : 'tensor([[     2,   6571,    708,    692, 235265]], device='cuda:0')'
Starting prompt: 'Who are you.'

Chosen Token    | Probability  | Top Alternatives Considered
----------------------------------------------------------------------
''              | 36.5      % | '' (14.3%), 'What' (11.9%), '' (7.7%)
'I'             | 58.2      % | 'My' (11.5%), 'Hi' (11.5%), 'Hello' (6.9%)
'am'            | 87.5      % | ''' (9.2%), 'don' (1.6%), '’' (0.3%)
'a'             | 50.8      % | 'Gemma' (39.6%), 'an' (3.7%), 'Bard' (3.7%)
'large'         | 96.9      % | 'helpful' (0.7%), 'language' (0.7%), 'Large' (0.5%)


In [3]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch is not detecting an available CUDA device or was not compiled with CUDA support.")

CUDA available: True
CUDA version: 12.8
Current device: 0
Device name: Tesla T4


If `CUDA available` is `False`, it means PyTorch was likely installed without CUDA support in this environment. In Colab, often changing the runtime type to GPU and then restarting the runtime (Runtime -> Restart runtime) can sometimes trigger the correct PyTorch installation. If not, you might need to manually install the CUDA-enabled PyTorch version (e.g., using `!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121` for CUDA 12.1, adjusting the CUDA version as needed, and then re-running the `transformers` and `accelerate` installs).